# Notebook 3 – Missing Value Handling

## 1. What are Missing Values?

### Concept / Explanation

A **missing value** means that a value is not available, not recorded, or unknown for a particular observation.

In Pandas, missing values are commonly represented as `NaN`.

`NaN` means **Not a Number**, but it is commonly used to represent missing data.

### Simple Example

Suppose we have student information:

```text
Student    Study_Hours    Attendance
A          5              90
B          7              95
C          NaN            88
D          4              NaN
```

Student C has a missing `Study_Hours`, and Student D has a missing `Attendance`.

### Real-World Example

A hospital may have patient information:

```text
Patient    Age    Blood_Pressure
P1         45     120
P2         NaN    130
P3         52     NaN
```

The patient's age or blood pressure may not have been recorded.

### Business Example

A company may maintain customer information:

```text
Customer    Age    Income
A           25     40000
B           NaN    45000
C           32     NaN
```

A customer may not have provided their age or income.

### AI/ML Use Case

Suppose we want to build a machine-learning model to predict customer churn:

```text
Age    Income    Tenure    Churn
25     40000     2         No
32     NaN       4         Yes
28     35000     NaN       No
```

Machine-learning algorithms may not be able to directly work with missing values, depending on the algorithm.

Therefore, missing values must be detected and handled appropriately before or during model training.


### Important Point

**Missing value is not the same as zero.**

```text
Age = 0
```

means the value is actually zero.

```text
Age = NaN
```

means the value is unknown or unavailable.

Therefore, we should **not blindly replace every missing value with the mean**. The correct method depends on the type of data and why the value is missing.


In [1]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["A", "B", "C", "D"],
    "Age": [25, np.nan, 30, 28],
    "Study_Hours": [5, 7, np.nan, 4]
})

print(df)



  Name   Age  Study_Hours
0    A  25.0          5.0
1    B   NaN          7.0
2    C  30.0          NaN
3    D  28.0          4.0


# 2. Why Missing Values Occur

### Concept / Explanation

Missing values can occur for many different reasons. Understanding **why the data is missing** is important because the reason can help us decide how to handle it.

Common reasons include:

* Data was not collected.
* User did not provide the information.
* Measurement failed.
* Human error occurred during data entry.
* System or technical problems occurred.
* A value was not applicable to a particular record.
* Data was lost during data transfer or merging.

### Simple Example

A student dataset:

```text
Student    Age    Study_Hours
A          20     5
B          21     NaN
C          22     6
```

Student B's study hours may be missing because the student did not provide the information.

### Real-World Example

In a hospital:

```text
Patient    Age    Blood_Pressure
P1         45     120
P2         50     NaN
P3         38     125
```

P2's blood pressure may be missing because the measurement was not taken.

### Business Example

An online shopping company collects customer information:

```text
Customer    Age    Income
A           25     40000
B           NaN    50000
C           30     NaN
```

Possible reasons:

* Customer B did not provide their age.
* Customer C did not provide their income.
* The information may have failed to transfer from another system.

### AI/ML Use Case

Suppose a company builds a model to predict customer churn.

Some customers may have missing `Income` values because they did not provide their income.

```text
Age    Income    Tenure    Churn
25     40000     2         No
32     NaN       4         Yes
28     35000     3         No
```

Before training the model, we need to understand **why `Income` is missing**.

If high-income customers are more likely to leave the income field blank, simply replacing missing values with the overall mean could introduce bias.


The important question is:

**Why are these values missing?**

Understanding the reason helps us choose the appropriate missing-value handling technique later.


In [3]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer": ["A", "B", "C", "D"],
    "Age": [25, np.nan, 30, 28],
    "Income": [40000, 50000, np.nan, 45000]
})

print(df)


  Customer   Age   Income
0        A  25.0  40000.0
1        B   NaN  50000.0
2        C  30.0      NaN
3        D  28.0  45000.0



Here, we can see missing values in `Age` and `Income`.

# 3. MCAR – Missing Completely At Random

### Concept / Explanation

**MCAR** means **Missing Completely At Random**.

A value is MCAR when the reason for the missing value has **no relationship with the missing value itself or any other variable in the dataset**.

In simple words:

> The data is missing purely by chance.

### Simple Example

Suppose a student dataset contains:

```text
Student    Age    Study_Hours
A          20     5
B          21     NaN
C          22     6
D          20     4
```

If Student B's study hours are missing because of a random technical error while entering the data, this can be considered **MCAR**.

The missingness is not related to the student's age, study hours, or any other information.

### Real-World Example

A hospital system accidentally fails to record blood pressure for a few randomly selected patients because of a temporary machine failure.

The missing values are unrelated to:

* Patient age
* Patient gender
* Blood pressure
* Disease
* Other patient characteristics

This can be an example of **MCAR**.

### Business Example

A retail company's data-entry system crashes randomly while recording customer information.

For a few randomly selected customers, the `Income` field is missing.

The missing values are not related to:

* Customer income
* Customer age
* Customer location
* Purchase amount

Therefore, the missingness may be considered **MCAR**.

### AI/ML Use Case

Suppose a machine-learning dataset has 1% missing values because of a random technical issue during data collection.

Since the missingness is unrelated to the features and target, removing those few rows may have very little effect on the model.

### Key Point

**MCAR = Missing because of pure chance.**

If the missingness is truly MCAR, common methods such as **dropping rows** can sometimes be reasonable, especially when the amount of missing data is very small.

However, we should not assume that every missing value is MCAR without investigating the data.


In [5]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Student": ["A", "B", "C", "D", "E"],
    "Age": [20, 21, 22, 20, 23],
    "Study_Hours": [5, np.nan, 6, 4, 7]
})

print(df)


  Student  Age  Study_Hours
0       A   20          5.0
1       B   21          NaN
2       C   22          6.0
3       D   20          4.0
4       E   23          7.0


Here, we are assuming that the missing `Study_Hours` value occurred because of a **random recording problem**.

# 4. MAR – Missing At Random

### Concept / Explanation

**MAR** means **Missing At Random**.

A value is MAR when the probability of a value being missing is related to **other observed variables in the dataset**, but not directly to the missing value itself after considering those observed variables.

In simple words:

> The missingness can be explained by information that we already have.

### Simple Example

Suppose we have student data:

```text
Student    Age    Study_Hours
A          20     5
B          21     NaN
C          22     6
D          20     NaN
```

Assume students with certain `Age` groups are more likely to leave `Study_Hours` blank.

For example, younger students may be less likely to report their study hours.

Here, the missingness of `Study_Hours` is related to the observed `Age`.

This can be an example of **MAR**.

### Real-World Example

A hospital collects patient information:

```text
Patient    Age    Gender    Blood_Pressure
P1         25     Male      120
P2         70     Female    NaN
P3         30     Male      125
P4         72     Female    NaN
```

Suppose older patients are more likely to have missing blood-pressure records because of a particular recording process.

Here, the missingness of `Blood_Pressure` is related to the observed `Age`.

Therefore, it can be considered **MAR**.

### Business Example

An e-commerce company collects customer information:

```text
Customer    Age    Membership    Income
A           25     Basic         30000
B           45     Premium       NaN
C           28     Basic         28000
D           50     Premium       NaN
```

Suppose premium customers are more likely to have missing `Income` because the company collects income information differently for that membership group.

The missingness of `Income` is related to the observed `Membership`.

This is an example of **MAR**.

### AI/ML Use Case

Suppose we are building a model to predict customer spending.

`Income` contains missing values, and the probability of missing income depends on observed variables such as:

* Age
* Membership type
* Location

We can use these available variables to help estimate the missing income values.

Techniques such as **group-based imputation, KNN imputation, or iterative imputation** may be useful depending on the dataset.


### Key Point

**MAR = Missingness is related to other information that we can observe.**

For example:

```text
Observed variable → Membership
                         ↓
                  Missing Income
```

Understanding MAR is important because simply using the overall mean may ignore the relationship between variables and produce a poor imputation.


In [7]:
### Simple Python Example
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer": ["A", "B", "C", "D"],
    "Age": [25, 45, 28, 50],
    "Membership": ["Basic", "Premium", "Basic", "Premium"],
    "Income": [30000, np.nan, 28000, np.nan]
})

print(df)


  Customer  Age Membership   Income
0        A   25      Basic  30000.0
1        B   45    Premium      NaN
2        C   28      Basic  28000.0
3        D   50    Premium      NaN


Here, we assume the missing `Income` values are related to the observed `Membership` type.

# 5. MNAR – Missing Not At Random

### Concept / Explanation

**MNAR** means **Missing Not At Random**.

A value is MNAR when the probability of a value being missing is related to **the missing value itself or information that is not observed**.

In simple words:

> The reason the value is missing is connected to the value that is missing.

### Simple Example

Suppose we have student data:

```text
Student    Study_Hours
A          5
B          NaN
C          6
D          4
```

Suppose students who study for **very long hours** are less likely to report their study hours.

Here, the missingness is related to the actual `Study_Hours` value.

This can be an example of **MNAR**.

### Real-World Example

Suppose a hospital asks patients to report their weight:

```text
Patient    Age    Weight
P1         25     65
P2         40     NaN
P3         35     70
```

If patients with higher body weight are more likely to avoid reporting their weight because they are uncomfortable sharing it, the missingness is related to the missing `Weight` itself.

This can be **MNAR**.

### Business Example

An e-commerce company asks customers for their annual income:

```text
Customer    Income
A           30000
B           NaN
C           45000
D           NaN
```

Suppose customers with very high incomes are more likely to avoid providing their income.

The probability of `Income` being missing depends on the actual income.

Therefore, the missingness may be **MNAR**.

### AI/ML Use Case

Suppose a financial company is building a model to predict customer income.

High-income customers are less likely to provide their income information.

If we simply replace missing income values with the overall mean, we may underestimate the incomes of those customers.

This can introduce **bias into the dataset and machine-learning model**.

MNAR situations require additional investigation and domain knowledge.


If we have a reason to believe that customers with very high incomes are more likely to leave `Income` blank, then the missingness may be **MNAR**.

### Key Point

**MNAR = The missingness is related to the value that is missing.**

A simple comparison:

```text
MCAR → Missing because of random chance

MAR  → Missing because of other observed information

MNAR  → Missing because of the missing value itself
```

MNAR is usually more difficult to handle because the information needed to understand the missingness may not be available in the dataset.


In [9]:
### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer": ["A", "B", "C", "D", "E"],
    "Income": [30000, 45000, np.nan, 35000, np.nan]
})

print(df)


  Customer   Income
0        A  30000.0
1        B  45000.0
2        C      NaN
3        D  35000.0
4        E      NaN


# 6. Detecting Missing Values

### Concept / Explanation

Before handling missing values, we first need to **identify where the missing values exist** in the dataset.

Pandas provides several useful functions:

* `isnull()` → identifies missing values.
* `isna()` → same as `isnull()`.
* `notnull()` → identifies non-missing values.
* `notna()` → same as `notnull()`.
* `sum()` → counts missing values.

### Simple Example

Suppose:

```text id="k4z9m1"
Name    Age    Salary
A       25     30000
B       NaN    40000
C       30     NaN
```

We can detect that:

* `Age` has 1 missing value.
* `Salary` has 1 missing value.

### Real-World Example

A hospital dataset may contain:

```text id="j7h2tr"
Patient    Age    BP    Sugar
P1         45     120   100
P2         NaN    130   110
P3         50     NaN   105
```

Before analysis, we need to identify which columns contain missing patient information.

### Business Example

An e-commerce company may have:

```text id="q3g5np"
Customer    Age    Income    Purchases
A           25     40000     5
B           NaN    45000     8
C           30     NaN       3
```

The data analyst should detect the missing values before calculating customer statistics or building reports.

### AI/ML Use Case

Before training a machine-learning model, we need to know:

* Which columns contain missing values?
* How many values are missing?
* Which rows contain missing values?

This helps us decide whether to:

* Drop rows
* Drop columns
* Impute values
* Use advanced imputation techniques

### Key Point

**Always detect missing values before deciding how to handle them.**

Do not immediately fill every missing value with the mean. First understand:

**Where are the missing values? → How many are missing? → Why are they missing? → Which handling technique is appropriate?**


In [16]:
### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["A", "B", "C", "D"],
    "Age": [25, np.nan, 30, 28],
    "Salary": [30000, 40000, np.nan, 35000]
})

print(df.isnull())




    Name    Age  Salary
0  False  False   False
1  False   True   False
2  False  False    True
3  False  False   False


In [13]:

#To count missing values:


print(df.isnull().sum())



Customer    0
Income      2
dtype: int64


In [10]:

print(df.isna().sum())


# `isna()` and `isnull()` give the same result.

Customer    0
Income      2
dtype: int64


# 7. Missing Value Percentage

### Concept / Explanation

Missing value percentage tells us **what percentage of values in each column are missing**.

Simply knowing the number of missing values is not always enough.

For example:

* 5 missing values out of 10 rows → **50% missing**
* 5 missing values out of 10,000 rows → **0.05% missing**

Therefore, percentage gives us a better understanding of the severity of missing data.

### Simple Example

Suppose we have 10 customer records and the `Age` column has 2 missing values.

```text
Missing Percentage = (2 / 10) × 100
                   = 20%
```

So, `Age` has **20% missing values**.

### Real-World Example

A hospital has 1,000 patient records.

If 50 patients have missing `Blood_Pressure`:

```text
Missing Percentage = (50 / 1000) × 100
                   = 5%
```

Only 5% of the blood-pressure data is missing.

### Business Example

An e-commerce company has 500 customer records:

```text
Age       → 10 missing
Income    → 150 missing
Location  → 5 missing
```

The percentages are:

```text
Age       → 2%
Income    → 30%
Location  → 1%
```

`Income` has a much larger missing-data problem than the other columns.

### AI/ML Use Case

Before building a machine-learning model, missing-value percentages help us decide whether a feature should be:

* Kept and imputed
* Dropped
* Investigated further

For example:

```text
Feature       Missing %
Age              2%
Income          30%
Education        1%
```

We may keep `Age` and `Education` and consider an appropriate imputation strategy for `Income`.

The decision should not be based only on a fixed percentage threshold; **business meaning and model requirements also matter**.

### Key Point

The basic formula is:

```text
Missing Percentage =
(Number of Missing Values / Total Number of Values) × 100
```

In Python:

```python id="1v8xpm"
df.isnull().mean() * 100
```

This is one of the first checks we should perform before choosing a missing-value handling technique.


In [17]:
### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["A", "B", "C", "D", "E"],
    "Age": [25, np.nan, 30, 28, np.nan],
    "Salary": [30000, 40000, np.nan, 35000, 45000]
})

missing_percentage = df.isnull().mean() * 100

print(missing_percentage)


#Here:

#* `Age` → **40% missing**
#* `Salary` → **20% missing**
#* `Name` → **0% missing**

Name       0.0
Age       40.0
Salary    20.0
dtype: float64


# 8. Dropping Rows

### Concept / Explanation

**Dropping rows** means removing complete records that contain missing values.

In Pandas, we commonly use:

```python
df.dropna()
```

This removes rows containing at least one missing value.

### Simple Example

Suppose we have:

```text
Student    Age    Study_Hours
A          20     5
B          21     NaN
C          22     6
```

If we drop rows containing missing values:

```text
Student    Age    Study_Hours
A          20     5
C          22     6
```

Student B is removed completely.

### Real-World Example

A hospital has 1,000 patient records, and only 5 records have missing information.

If those 5 records are not important for the analysis, we may remove those rows instead of estimating the missing values.

### Business Example

An e-commerce company has 10,000 customer records, but 20 records have incomplete information.

If these 20 records represent only a very small percentage of the dataset and are not important for the analysis, dropping those rows may be acceptable.

### AI/ML Use Case

Suppose a machine-learning dataset contains 10,000 records and only 20 rows have missing values.

Removing those 20 rows may be reasonable because we still retain **9,980 records** for training.

However, we should check whether those rows are randomly missing or contain an important customer segment.

### When to Use

Use row deletion when:

* Only a small percentage of rows contain missing values.
* The dataset is large enough to tolerate losing some records.
* The missing records are not important to the analysis.
* The missingness does not introduce significant bias.

### When Not to Use

Avoid dropping rows when:

* A large percentage of rows contain missing values.
* The dataset is small.
* The missing records contain important information.
* Removing rows could create bias.
* The missingness is related to a particular customer or patient group.

### Advantages

* Very simple.
* Easy to implement.
* Does not create artificial values.
* Preserves the original values of the remaining records.

### Limitations

* Causes loss of data.
* Can reduce the size of the training dataset.
* Can introduce bias if missingness is not random.
* May remove useful information.

### Key Point

**Do not automatically delete every row containing a missing value.**

First check **how many rows will be removed and why the values are missing**.


In [18]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["A", "B", "C", "D"],
    "Age": [25, np.nan, 30, 28],
    "Salary": [30000, 40000, np.nan, 35000]
})

print("Before:")
print(df)

df_dropped = df.dropna()

print("\nAfter:")
print(df_dropped)


Before:
  Name   Age   Salary
0    A  25.0  30000.0
1    B   NaN  40000.0
2    C  30.0      NaN
3    D  28.0  35000.0

After:
  Name   Age   Salary
0    A  25.0  30000.0
3    D  28.0  35000.0


# 9. Dropping Columns

### Concept / Explanation

**Dropping columns** means removing an entire feature from the dataset because it contains too many missing values or is not useful for the analysis.

In Pandas:

```python
df.drop(columns=["Column_Name"])
```

### Simple Example

Suppose:

```text
Student    Age    Study_Hours    Address
A          20     5              Chennai
B          21     6              NaN
C          22     NaN            NaN
D          23     7              NaN
```

If `Address` contains too many missing values and is not important for our analysis, we may remove the entire `Address` column.

### Real-World Example

A hospital dataset contains:

```text
Patient    Age    Blood_Pressure    Insurance_ID
P1         45     120               ABC01
P2         50     130               NaN
P3         38     125               NaN
P4         60     NaN               NaN
```

If `Insurance_ID` is missing for most patients and is not required for the analysis, we may drop that column.

### Business Example

An e-commerce company has:

```text
Customer    Age    Income    Referral_Code
A           25     40000    ABC
B           30     45000    NaN
C           28     35000    NaN
D           32     50000    NaN
```

If `Referral_Code` is missing for most customers and is not useful for the business analysis, we could remove it.

### AI/ML Use Case

Suppose a machine-learning dataset contains:

```text
Feature          Missing %
Age                 2%
Income             10%
Referral_Code      95%
```

`Referral_Code` has 95% missing values.

If it does not provide meaningful predictive information, dropping the feature may be better than trying to fill 95% of its values.

opped)
```

### When to Use

Use column deletion when:

* A column has a very high percentage of missing values.
* The feature is not important for the analysis.
* Reliable imputation is not possible.
* The feature provides little or no useful information.

### When Not to Use

Do not drop a column just because it has missing values when:

* The feature is highly important.
* The column contains useful predictive information.
* A suitable imputation method is available.
* The missing values are relatively small in number.

### Advantages

* Simple and fast.
* Removes the missing-value problem for that feature.
* Reduces the number of features.
* Can simplify a machine-learning model.

### Limitations

* Important information may be lost.
* Removing an important feature can reduce model performance.
* A high missing percentage does not automatically mean the feature is useless.
* Business/domain knowledge should be considered before removing it.

### Key Point

**High missing percentage ≠ automatically drop the column.**

Before dropping a feature, ask:

**How much data is missing? → Is the feature important? → Can it be reliably imputed? → Does removing it affect the business or ML objective?**


In [21]:
### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["A", "B", "C", "D"],
    "Age": [25, 30, 28, 32],
    "Income": [30000, 40000, 35000, 45000],
    "Referral_Code": ["A12", np.nan, np.nan, np.nan]
})

print("Before:")
print(df)

df_dropped = df.drop(columns=["Referral_Code"])

print("\nAfter:")
print(df_dropped)

Before:
  Name  Age  Income Referral_Code
0    A   25   30000           A12
1    B   30   40000           NaN
2    C   28   35000           NaN
3    D   32   45000           NaN

After:
  Name  Age  Income
0    A   25   30000
1    B   30   40000
2    C   28   35000
3    D   32   45000


# 10. Mean Imputation

### Concept / Explanation

**Mean imputation** means replacing missing numerical values with the **mean (average)** of the available values in that column.

Formula:

```text
Mean = Sum of values / Number of available values
```

For example:

```text
Age = 20, 25, NaN, 30, 35

Mean = (20 + 25 + 30 + 35) / 4
     = 27.5
```

The missing value is replaced with `27.5`.

### Simple Example

Suppose:

```text
Student    Marks
A          70
B          80
C          NaN
D          90
```

Mean:

```text
(70 + 80 + 90) / 3 = 80
```

After mean imputation:

```text
Student    Marks
A          70
B          80
C          80
D          90
```

### Real-World Example

A hospital has patient ages:

```text
Age = 25, 30, 35, NaN, 40
```

If the age distribution is reasonably balanced and there are no major outliers, the mean age can be used to replace the missing value.

### Business Example

A company has customer purchase amounts:

```text
Purchase_Amount = 1000, 1200, 1500, NaN, 1300
```

If the values are reasonably distributed without extreme purchases, the average purchase amount can be used for the missing value.

### AI/ML Use Case

Suppose a machine-learning dataset has a numerical feature called `Income` with a small number of missing values.

If the income distribution is approximately symmetric and does not contain significant outliers, mean imputation can provide a simple way to prepare the data for model training.


### When to Use

Use mean imputation when:

* The variable is numerical.
* The data is approximately normally distributed or symmetric.
* There are few missing values.
* There are no significant outliers.
* A simple imputation method is sufficient.

### When Not to Use

Do not use mean imputation when:

* The data is highly skewed.
* There are significant outliers.
* The variable is categorical.
* A large percentage of values are missing.
* Different groups have very different means.

For example, if salaries are:

```text
20,000
25,000
30,000
35,000
5,00,000
```

the extreme salary can pull the mean upward and produce an unrealistic replacement.

### Advantages

* Very simple.
* Fast to calculate.
* Easy to implement.
* Keeps the same number of rows.
* Useful for approximately symmetric numerical data.

### Limitations

* Can distort the distribution.
* Reduces natural variability.
* Can be strongly affected by outliers.
* Can weaken relationships between variables.
* May introduce bias when missingness is not random.

### Key Point

**Do not use mean imputation automatically.**

Always check the distribution and outliers first.

If the data is skewed or contains outliers, **median imputation may be a better choice**.


# 11. Median Imputation

### Concept / Explanation

**Median imputation** means replacing missing numerical values with the **median** of the available values.

The median is the **middle value** after sorting the data.

Example:

```text id="w5m6d8"
Age = 20, 25, 30, 35, 100
```

The median is:

```text id="ryq1n3"
30
```

If a value is missing, we can replace it with `30`.

Median is generally less affected by extreme values than the mean.

### Simple Example

Suppose:

```text id="k4d1u7"
Student    Marks
A          60
B          70
C          NaN
D          80
E          95
```

Sorted values:

```text id="x9o4c2"
60, 70, 80, 95
```

The median is:

```text id="h7q2sm"
(70 + 80) / 2 = 75
```

So the missing value becomes `75`.

### Real-World Example

Suppose a hospital has patient ages:

```text id="8k6zsp"
Age = 25, 30, 35, 40, 85, NaN
```

The value `85` is relatively high compared with the other ages.

The mean can be influenced by this extreme value, while the median is more resistant to it.

Therefore, median imputation may be more appropriate.

### Business Example

An e-commerce company records customer purchase amounts:

```text id="v7j8q4"
Purchase = 500, 700, 800, 1000, 50000, NaN
```

The `50000` purchase is an extreme value.

Using the mean would be strongly affected by this large purchase.

The median provides a more representative typical purchase amount.

### AI/ML Use Case

Suppose a machine-learning dataset contains a numerical feature such as `Annual_Income`.

The distribution is highly skewed because a small number of customers have extremely high incomes.

For example:

```text id="f4h1g8"
30000, 35000, 40000, 45000, 500000, NaN
```

Median imputation is often preferable because it is less sensitive to the extreme income value.


### When to Use

Use median imputation when:

* The variable is numerical.
* The data is skewed.
* The dataset contains outliers.
* There are relatively few missing values.
* A simple and robust imputation method is required.

### When Not to Use

Avoid median imputation when:

* The variable is categorical.
* The missingness contains important information.
* The data has meaningful group differences that should be considered.
* A more suitable model-based or group-based method is available.

### Advantages

* Simple and fast.
* Less affected by outliers than the mean.
* Works well with skewed numerical data.
* Easy to implement.

### Limitations

* Reduces natural variability.
* Can distort the distribution.
* May weaken relationships between variables.
* Does not use information from other features.
* Can be inappropriate when different groups have very different medians.

### Key Point

**Mean vs Median**

```text id="h6by2n"
Symmetric data + few outliers → Mean may work well

Skewed data / outliers          → Median is often better
```

Therefore, **median is not automatically better than mean**. The choice depends on the distribution and characteristics of the data.


In [22]:

### Simple Python Example


import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["A", "B", "C", "D", "E"],
    "Purchase": [500, 700, np.nan, 1000, 50000]
})

print("Before:")
print(df)

median_value = df["Purchase"].median()

df["Purchase"] = df["Purchase"].fillna(median_value)

print("\nAfter:")
print(df)


Before:
  Name  Purchase
0    A     500.0
1    B     700.0
2    C       NaN
3    D    1000.0
4    E   50000.0

After:
  Name  Purchase
0    A     500.0
1    B     700.0
2    C     850.0
3    D    1000.0
4    E   50000.0


# 12. Mode Imputation

### Concept / Explanation

**Mode imputation** means replacing missing values with the **most frequently occurring value** in a column.

The mode is mainly useful for **categorical data**, such as:

* Gender
* Department
* City
* Product Category
* Payment Method

Example:

```text
Department = IT, HR, IT, Sales, IT, NaN
```

The mode is:

```text
IT
```

So the missing value can be replaced with `IT`.

### Simple Example

Suppose:

```text
Student    Department
A          IT
B          HR
C          IT
D          NaN
E          IT
```

`IT` occurs most frequently.

Therefore:

```text
Student    Department
A          IT
B          HR
C          IT
D          IT
E          IT
```

### Real-World Example

A hospital records patient departments:

```text
Patient    Department
P1         Cardiology
P2         General
P3         Cardiology
P4         NaN
P5         Cardiology
```

`Cardiology` occurs most frequently.

The missing department can be replaced with the mode if there is a reasonable assumption that the missingness is not systematically associated with another department.

### Business Example

An e-commerce company records customers' preferred payment methods:

```text
Customer    Payment_Method
A           UPI
B           Card
C           UPI
D           NaN
E           UPI
```

The mode is `UPI`.

The missing payment method can therefore be replaced with `UPI`.

### AI/ML Use Case

Suppose a machine-learning dataset contains a categorical feature:

```text
Payment_Method
UPI
Card
UPI
NaN
UPI
```

Many machine-learning algorithms cannot directly process missing categorical values.

If only a small number of values are missing, mode imputation can be a simple preprocessing approach.


### When to Use

Use mode imputation when:

* The feature is categorical.
* Only a small percentage of values are missing.
* One category clearly dominates the column.
* A simple imputation method is appropriate.

### When Not to Use

Avoid mode imputation when:

* There are many missing values.
* Categories have similar frequencies.
* The missingness itself has business meaning.
* Replacing missing values with the most common category could create significant bias.

### Advantages

* Simple and fast.
* Easy to understand.
* Works well for categorical variables.
* Does not require complex calculations.

### Limitations

* Can increase the frequency of the most common category.
* Can distort the original category distribution.
* May introduce bias.
* Does not consider relationships with other features.

### Key Point

```text
Numerical data → Mean / Median may be appropriate

Categorical data → Mode may be appropriate
```

But mode should **not automatically be used for every categorical missing value**. We should first understand the missingness and the distribution of categories.


In [23]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer": ["A", "B", "C", "D", "E"],
    "Payment_Method": ["UPI", "Card", "UPI", np.nan, "UPI"]
})

print("Before:")
print(df)

mode_value = df["Payment_Method"].mode()[0]

df["Payment_Method"] = df["Payment_Method"].fillna(mode_value)

print("\nAfter:")
print(df)


Before:
  Customer Payment_Method
0        A            UPI
1        B           Card
2        C            UPI
3        D            NaN
4        E            UPI

After:
  Customer Payment_Method
0        A            UPI
1        B           Card
2        C            UPI
3        D            UPI
4        E            UPI


# 13. Constant Value Imputation

### Concept / Explanation

**Constant value imputation** means replacing missing values with a **specific value that we choose**, instead of calculating the mean, median, or mode.

Common constant values include:

```text
0
-1
"Unknown"
"Not Provided"
"Missing"
```

The chosen value should have a meaningful interpretation for the dataset.

### Simple Example

Suppose:

```text
Name    City
A       Chennai
B       NaN
C       Bangalore
```

We can replace the missing city with `"Unknown"`:

```text
Name    City
A       Chennai
B       Unknown
C       Bangalore
```

### Real-World Example

A hospital collects patients' insurance information:

```text
Patient    Insurance
P1         ABC
P2         NaN
P3         XYZ
```

If the patient did not provide insurance information, we can use:

```text
"Not Provided"
```

instead of pretending that the patient belongs to a particular insurance company.

### Business Example

An e-commerce company has a `Discount_Code` column:

```text
Customer    Discount_Code
A           SAVE10
B           NaN
C           FEST20
```

A missing discount code could be replaced with:

```text
"No Discount"
```

This gives the missing value a meaningful business interpretation.

### AI/ML Use Case

Suppose a machine-learning dataset contains a categorical feature:

```text
Education
Graduate
Postgraduate
NaN
Graduate
```

We can replace the missing value with `"Unknown"`.

This preserves the information that the education level was **not available**, rather than incorrectly assigning the most common education category.



### When to Use

Use constant value imputation when:

* A meaningful default value exists.
* `"Unknown"` or `"Not Provided"` has business meaning.
* You want to preserve the fact that the value was missing.
* The feature is categorical and missingness itself may be informative.
* A numerical constant such as `0` has a genuine meaning.

### When Not to Use

Avoid it when:

* The chosen constant has no meaningful interpretation.
* `0` could be confused with an actual zero.
* Replacing values with one constant would create a large artificial group.
* The feature requires a more accurate estimation method.

For example, replacing missing `Age` with `0` is usually inappropriate because `0` would not represent an unknown adult customer's age.

### Advantages

* Very simple and fast.
* Easy to understand.
* Preserves missingness information when using values such as `"Unknown"`.
* Useful for categorical business data.

### Limitations

* Can create an artificial category.
* Can distort the distribution.
* A poor choice of constant can introduce bias.
* Numerical constants can significantly affect statistics and model performance.

### Key Point

**Choose a constant because it has meaning, not simply because it is convenient.**

For example:

```text
Missing City       → "Unknown"        ✓
Missing Discount   → "No Discount"    ✓
Missing Age        → 0                ✗ usually inappropriate
```


In [24]:
### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer": ["A", "B", "C", "D"],
    "City": ["Chennai", np.nan, "Bangalore", np.nan]
})

print("Before:")
print(df)

df["City"] = df["City"].fillna("Unknown")

print("\nAfter:")
print(df)


Before:
  Customer       City
0        A    Chennai
1        B        NaN
2        C  Bangalore
3        D        NaN

After:
  Customer       City
0        A    Chennai
1        B    Unknown
2        C  Bangalore
3        D    Unknown


# 14. Forward Fill

### Concept / Explanation

**Forward fill (ffill)** replaces a missing value with the **previous available value**.

In Pandas:

```python
df["column"].ffill()
```

For example:

```text
Time    Temperature
10:00   25
11:00   NaN
12:00   27
```

After forward fill:

```text
Time    Temperature
10:00   25
11:00   25
12:00   27
```

The missing value at `11:00` is filled using the previous value `25`.

### Simple Example

```text
Day    Stock_Price
Mon    100
Tue    NaN
Wed    105
```

Forward fill:

```text
Day    Stock_Price
Mon    100
Tue    100
Wed    105
```

### Real-World Example

A temperature monitoring system records:

```text
Time    Temperature
10 AM   30
11 AM   NaN
12 PM   32
```

If the sensor temporarily fails at 11 AM, we may use the previous reading as an estimate:

```text
10 AM   30
11 AM   30
12 PM   32
```

### Business Example

A retail store records its inventory level:

```text
Time    Stock
10 AM   100
11 AM   NaN
12 PM   80
```

If the 11 AM reading is missing because of a temporary system issue, forward fill can carry the last known stock value forward.

### AI/ML Use Case

Forward fill is commonly useful when working with **time-series data**, such as:

* Stock prices
* Sensor readings
* Website traffic
* Electricity usage
* IoT data

For example, if a sensor temporarily fails, the previous observation can be carried forward.

### When to Use

Use forward fill when:

* Data has a meaningful order.
* Data is time-series or sequential.
* The previous value is reasonably expected to remain valid for the next observation.
* Values change relatively slowly.

### When Not to Use

Avoid forward fill when:

* Rows have no meaningful order.
* Values change rapidly.
* The previous value is not representative of the next value.
* Carrying an old value forward could create misleading information.

For example, using yesterday's stock price to fill a missing value from several months later would not be appropriate.

### Advantages

* Very simple and fast.
* Useful for time-series data.
* Preserves the previous known value.
* Does not require statistical calculations.

### Limitations

* Can propagate incorrect values.
* Can create long sequences of repeated values.
* Does not consider future observations.
* Can introduce bias if the previous value is not representative.

### Key Point

**Forward Fill = Use the previous available value.**

```text
100
NaN    → 100
NaN    → 100
105
```

Use it mainly when the **order of observations matters**.


In [25]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Time": ["10 AM", "11 AM", "12 PM", "1 PM"],
    "Temperature": [30, np.nan, 32, np.nan]
})

print("Before:")
print(df)

df["Temperature"] = df["Temperature"].ffill()

print("\nAfter:")
print(df)



Before:
    Time  Temperature
0  10 AM         30.0
1  11 AM          NaN
2  12 PM         32.0
3   1 PM          NaN

After:
    Time  Temperature
0  10 AM         30.0
1  11 AM         30.0
2  12 PM         32.0
3   1 PM         32.0


# 15. Backward Fill

### Concept / Explanation

**Backward fill (bfill)** replaces a missing value with the **next available value**.

In Pandas:

```python
df["column"].bfill()
```

It is the opposite of forward fill.

```text
Forward Fill  → Previous value
Backward Fill → Next value
```

### Simple Example

Suppose:

```text
Day    Stock_Price
Mon    100
Tue    NaN
Wed    105
```

Using backward fill:

```text
Day    Stock_Price
Mon    100
Tue    105
Wed    105
```

The missing Tuesday value is filled using Wednesday's value.

### Real-World Example

A temperature sensor records:

```text
Time    Temperature
10 AM   30
11 AM   NaN
12 PM   32
```

If the missing 11 AM reading is expected to be close to the next available reading, we can use backward fill:

```text
10 AM   30
11 AM   32
12 PM   32
```

### Business Example

An e-commerce system records product inventory:

```text
Time    Stock
10 AM   100
11 AM   NaN
12 PM   80
```

If the 11 AM value is missing because of a temporary recording problem and the next known value is considered more appropriate, backward fill can be used.

### AI/ML Use Case

Backward fill can be useful with ordered or time-series data when the **next observation** is more appropriate for estimating a missing value.

For example, it can be used for:

* Sensor data
* Time-series measurements
* Operational monitoring
* Sequential records

However, for forecasting or real-time prediction, using future information may cause **data leakage**.

Notice that the last value remains `NaN` because there is **no next available value** to use.

### When to Use

Use backward fill when:

* Data has a meaningful order.
* The next observation is a reasonable estimate for the missing value.
* You are working with sequential or time-series data.
* Future observations are legitimately available for the task.

### When Not to Use

Avoid backward fill when:

* The data has no meaningful order.
* Future information would not have been available at prediction time.
* The variable changes rapidly.
* Using future values could cause data leakage in an ML model.

### Advantages

* Simple and fast.
* Useful for ordered data.
* Does not require statistical calculations.
* Can be useful when the next observation is more representative.

### Limitations

* Can propagate incorrect values backward.
* Does not work for missing values at the end of the dataset.
* Can create repeated values.
* In ML/time-series prediction, it can introduce **data leakage** if future information is used improperly.

### Key Point

**Backward Fill = Use the next available value.**

```text
100
NaN    → 105
105
```

Use it carefully, especially in machine learning, because **future information should not be used to predict the past or simulate real-time predictions**.


In [26]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Time": ["10 AM", "11 AM", "12 PM", "1 PM"],
    "Temperature": [30, np.nan, 32, np.nan]
})

print("Before:")
print(df)

df["Temperature"] = df["Temperature"].bfill()

print("\nAfter:")
print(df)



Before:
    Time  Temperature
0  10 AM         30.0
1  11 AM          NaN
2  12 PM         32.0
3   1 PM          NaN

After:
    Time  Temperature
0  10 AM         30.0
1  11 AM         32.0
2  12 PM         32.0
3   1 PM          NaN


# 16. Interpolation

### Concept / Explanation

**Interpolation** estimates a missing value using the values **before and after it**.

Unlike forward fill or backward fill, interpolation does not simply copy an existing value. It calculates an estimated value between known values.

For example:

```text
Time    Temperature
10 AM   20
11 AM   NaN
12 PM   30
```

The missing value is between `20` and `30`.

Linear interpolation estimates:

```text
(20 + 30) / 2 = 25
```

So:

```text
Time    Temperature
10 AM   20
11 AM   25
12 PM   30
```

### Simple Example

Suppose:

```text
Day    Sales
1      100
2      NaN
3      300
```

The missing value lies between 100 and 300.

Interpolation gives:

```text
Day    Sales
1      100
2      200
3      300
```

### Real-World Example

A temperature sensor records:

```text
Time    Temperature
10 AM   20
11 AM   NaN
12 PM   30
```

If temperature changes gradually, interpolation can estimate the missing 11 AM temperature as `25`.

### Business Example

A company records daily sales:

```text
Day    Sales
Monday     1000
Tuesday    NaN
Wednesday  1400
```

If sales are expected to change gradually, interpolation can estimate Tuesday's sales:

```text
Tuesday = 1200
```

This can be useful for analyzing trends.

### AI/ML Use Case

Interpolation can be useful for **ordered numerical data**, especially:

* Sensor data
* Time-series data
* Temperature readings
* Stock/financial measurements
* Production measurements

It can provide a more realistic estimate than simply replacing every missing value with the overall mean.

The missing value between `100` and `300` is estimated as `200`.

### When to Use

Use interpolation when:

* Data has a meaningful order.
* Values change gradually or continuously.
* There are known values before and after the missing value.
* You are working with time-series or sequential numerical data.

### When Not to Use

Avoid interpolation when:

* The data is categorical.
* Values change randomly or abruptly.
* There is no meaningful order.
* The missing value cannot reasonably be estimated from neighboring observations.
* Future values should not be used for a real-time ML prediction.

### Advantages

* Uses surrounding values.
* Can preserve the trend of the data.
* Often more realistic than using a single global mean.
* Useful for continuous numerical measurements.

### Limitations

* Assumes a meaningful relationship between neighboring values.
* Can produce inaccurate estimates when data changes abruptly.
* Requires ordered data.
* Can cause data leakage if future observations are used improperly in ML.

### Key Point

```text
Forward Fill  → Copy previous value
Backward Fill → Copy next value
Interpolation → Estimate between known values
```

For example:

```text
100
NaN    → 200
300
```

Interpolation estimates the missing value based on the surrounding observations.


In [27]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Day": [1, 2, 3, 4],
    "Sales": [100, np.nan, 300, 400]
})

print("Before:")
print(df)

df["Sales"] = df["Sales"].interpolate()

print("\nAfter:")
print(df)


Before:
   Day  Sales
0    1  100.0
1    2    NaN
2    3  300.0
3    4  400.0

After:
   Day  Sales
0    1  100.0
1    2  200.0
2    3  300.0
3    4  400.0


# 17. Group-Based Imputation

### Concept / Explanation

**Group-based imputation** means filling missing values using information from the **same group** instead of using one value for the entire dataset.

For example, instead of using the overall average salary for all employees, we can calculate the average salary separately for each department.

```text
Department    Salary
IT            50000
IT            NaN
HR            40000
HR            45000
```

For the missing IT salary, we use the **IT group's average**, not the overall average.

### Simple Example

Suppose:

```text
Department    Salary
IT            50000
IT            60000
IT            NaN
HR            40000
HR            45000
HR            NaN
```

IT average:

```text
(50000 + 60000) / 2 = 55000
```

HR average:

```text
(40000 + 45000) / 2 = 42500
```

Therefore:

```text
Department    Salary
IT            50000
IT            60000
IT            55000
HR            40000
HR            45000
HR            42500
```

### Real-World Example

A hospital has patient data:

```text
Department    Age
Cardiology    50
Cardiology    NaN
General       30
General       35
```

Patients in different departments may have different typical ages.

Instead of using the overall median age, we can calculate the median separately for each department.

### Business Example

A company has employee salaries:

```text
Department    Salary
IT            60000
IT            65000
IT            NaN
HR            40000
HR            45000
HR            NaN
```

Using the overall salary average could be misleading because IT and HR have different salary ranges.

Group-based imputation uses the appropriate department's typical salary.

### AI/ML Use Case

Suppose we are predicting customer spending.

Customers belong to different membership groups:

```text
Membership    Spending
Basic         1000
Basic         1200
Basic         NaN
Premium       5000
Premium       6000
Premium       NaN
```

Basic and Premium customers have very different spending patterns.

Using one overall mean could distort the missing values.

Group-based imputation can estimate missing spending separately for each membership group.


Here:

* Missing IT salary → IT mean = `55000`
* Missing HR salary → HR mean = `42500`

### When to Use

Use group-based imputation when:

* Different groups have different distributions.
* A relevant grouping variable is available.
* Group membership is meaningful.
* There are enough observations within each group.

Examples of groups:

* Department
* Gender
* Location
* Customer segment
* Product category
* Membership type

### When Not to Use

Avoid it when:

* Groups have very few observations.
* The grouping variable is not meaningful.
* Groups have very similar distributions.
* Grouping creates unstable estimates.
* The grouping variable itself contains many missing values.

### Advantages

* More accurate than using one overall mean or median when groups differ.
* Preserves group-level patterns.
* Simple to implement.
* Easy to explain to business users.

### Limitations

* Requires a meaningful grouping variable.
* Small groups can produce unreliable estimates.
* Can become complex with many groups.
* Does not automatically capture relationships between multiple features.

### Key Point

**Overall imputation:**

```text
All customers → One common value
```

**Group-based imputation:**

```text
Basic customers   → Basic group value
Premium customers → Premium group value
```

Therefore, group-based imputation can be much better than blindly using the overall mean when different groups behave differently.


In [28]:

### Simple Python Example

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Department": ["IT", "IT", "IT", "HR", "HR", "HR"],
    "Salary": [50000, 60000, np.nan, 40000, 45000, np.nan]
})

print("Before:")
print(df)

df["Salary"] = df.groupby("Department")["Salary"].transform(
    lambda x: x.fillna(x.mean())
)

print("\nAfter:")
print(df)


Before:
  Department   Salary
0         IT  50000.0
1         IT  60000.0
2         IT      NaN
3         HR  40000.0
4         HR  45000.0
5         HR      NaN

After:
  Department   Salary
0         IT  50000.0
1         IT  60000.0
2         IT  55000.0
3         HR  40000.0
4         HR  45000.0
5         HR  42500.0


# 18. KNN Imputation

### Concept / Explanation

**KNN Imputation** means **K-Nearest Neighbors Imputation**.

Instead of replacing a missing value with one overall value such as the mean or median, KNN looks for **similar rows** and uses their values to estimate the missing value.

The basic idea is:

> Similar records are likely to have similar values.

For example, if a customer's `Income` is missing, KNN can find other customers with similar `Age`, `Spending`, and `Tenure`, then use their income values to estimate the missing income.

### Simple Example

Suppose:

```text
Student    Study_Hours    Attendance    Marks
A          5              90             75
B          6              92             80
C          5              88             NaN
D          2              60             40
```

Student C has missing `Marks`.

Students A and B are more similar to C because their study hours and attendance are similar.

KNN can use those similar students to estimate C's missing marks.

### Real-World Example

A hospital has:

```text
Patient    Age    Weight    Blood_Pressure
P1         30     65        120
P2         32     67        122
P3         31     66        NaN
P4         70     90        150
```

P3 is more similar to P1 and P2 than P4.

KNN can use the blood-pressure values of similar patients to estimate P3's missing value.

### Business Example

An e-commerce company has:

```text
Customer    Age    Spending    Visits    Income
A           25     5000        10        30000
B           27     5500        12        35000
C           26     5200        11        NaN
D           55     1000        3         70000
```

Customer C is more similar to A and B than D.

KNN can use similar customers to estimate C's missing `Income`.

### AI/ML Use Case

KNN imputation can be useful when several features are related to each other.

For example, when predicting customer income, features such as:

* Age
* Spending
* Number of purchases
* Tenure

may help identify similar customers.

KNN can use these relationships to estimate missing values.

Here:

```python id="qld5h9"
KNNImputer(n_neighbors=2)
```

means that the algorithm considers the **2 nearest/similar observations** when estimating missing values.

### When to Use

Use KNN imputation when:

* Multiple features are related.
* Similar observations exist in the dataset.
* The dataset is reasonably sized.
* You want to use relationships between features instead of one overall statistic.
* A more sophisticated method than mean or median is appropriate.

### When Not to Use

Avoid KNN imputation when:

* The dataset is extremely large and computational efficiency is important.
* Features are unrelated.
* There are too few similar observations.
* Features are not properly scaled.
* The dataset contains many missing values across many important features.

### Advantages

* Uses relationships between observations.
* Can produce more realistic estimates than simple mean/median imputation.
* Can work with multiple numerical features.
* Adapts to local patterns in the data.

### Limitations

* More computationally expensive than mean or median imputation.
* Sensitive to feature scaling.
* Choice of `k` affects the result.
* Can perform poorly when similar observations do not exist.
* Can become slow with very large datasets.

### Key Point

```text
Mean Imputation → Uses overall average

Median Imputation → Uses overall middle value

KNN Imputation → Uses similar observations
```

KNN is therefore more sophisticated, but **more complex does not automatically mean better**. The dataset and relationships between features should determine whether KNN is appropriate.


In [29]:
### Simple Python Example

import pandas as pd
import numpy as np

from sklearn.impute import KNNImputer

df = pd.DataFrame({
    "Age": [25, 27, 26, 55],
    "Spending": [5000, 5500, 5200, 1000],
    "Income": [30000, 35000, np.nan, 70000]
})

print("Before:")
print(df)

imputer = KNNImputer(n_neighbors=2)

df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

print("\nAfter:")
print(df_imputed)



Before:
   Age  Spending   Income
0   25      5000  30000.0
1   27      5500  35000.0
2   26      5200      NaN
3   55      1000  70000.0

After:
    Age  Spending   Income
0  25.0    5000.0  30000.0
1  27.0    5500.0  35000.0
2  26.0    5200.0  32500.0
3  55.0    1000.0  70000.0


# 19. Iterative Imputation

### Concept / Explanation

**Iterative Imputation** estimates missing values by using **relationships between multiple features**.

Instead of looking at only one column, it builds a model to predict the missing values using the other available columns.

The process is repeated several times, improving the estimates at each iteration.

Simple idea:

```text
Age + Experience → predict Salary
Salary + Experience → predict Age
Age + Salary → predict Experience
```

The process continues until the estimated values become reasonably stable.

### Simple Example

Suppose:

```text
Age    Experience    Salary
25     2             30000
30     5             NaN
35     8             50000
40     NaN           60000
```

`Salary` may have a relationship with `Age` and `Experience`.

Iterative imputation can use the available features to estimate the missing values rather than simply using the overall mean.

### Real-World Example

A hospital dataset contains:

```text
Age    Weight    Blood_Pressure    Cholesterol
30     65        120               180
40     NaN       130               200
50     80        NaN               220
```

These medical measurements may have relationships with each other.

Iterative imputation can use the available variables to estimate missing `Weight` and `Blood_Pressure`.

### Business Example

A company has employee information:

```text
Experience    Salary    Performance
2             30000     70
5             NaN       80
8             60000     90
```

Salary may be related to experience and performance.

Instead of using one overall salary average, iterative imputation can use the relationships between these variables to estimate the missing salary.

### AI/ML Use Case

Iterative imputation is useful when several features are related.

For example, in a customer dataset:

```text
Age
Income
Spending
Credit_Score
```

If some values are missing, the relationships between these features can help estimate the missing values.

It can be especially useful when simple methods such as mean or median imputation would lose important relationships between features.


The `IterativeImputer` uses the available features to estimate the missing values.

### When to Use

Use iterative imputation when:

* Multiple numerical features have relationships with each other.
* Missing values occur in more than one feature.
* Simple mean or median imputation is not sufficient.
* You want to preserve relationships between variables.
* The dataset is suitable for model-based imputation.

### When Not to Use

Avoid it when:

* The dataset is extremely large and a simple method is sufficient.
* Features have very weak relationships.
* You need a very simple and easily explainable approach.
* The dataset is too small to build reliable predictive relationships.
* The additional computational complexity is not justified.

### Advantages

* Uses relationships between multiple variables.
* Usually more informative than simple mean or median imputation.
* Can handle missing values in multiple features.
* Can preserve relationships between variables better.

### Limitations

* More computationally expensive.
* More complex than basic imputation methods.
* Results depend on the underlying relationships between features.
* Can introduce model-based bias.
* Requires careful validation to avoid poor estimates or data leakage.

### Key Point

```text
Mean/Median
     ↓
Use one column's statistics

KNN
     ↓
Use similar rows

Iterative Imputation
     ↓
Use relationships between multiple features
```

Iterative imputation is a **more advanced technique**, so it should not automatically replace simpler methods. Start with a simple, appropriate method and use advanced imputation when the dataset justifies it.


In [30]:

### Simple Python Example

import pandas as pd
import numpy as np

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df = pd.DataFrame({
    "Age": [25, 30, 35, 40],
    "Experience": [2, 5, 8, np.nan],
    "Salary": [30000, np.nan, 50000, 60000]
})

print("Before:")
print(df)

imputer = IterativeImputer(random_state=42)

df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

print("\nAfter:")
print(df_imputed)


Before:
   Age  Experience   Salary
0   25         2.0  30000.0
1   30         5.0      NaN
2   35         8.0  50000.0
3   40         NaN  60000.0

After:
    Age  Experience        Salary
0  25.0         2.0  30000.000000
1  30.0         5.0  40000.000003
2  35.0         8.0  50000.000000
3  40.0        11.0  60000.000000
